# Carbon dioxide removal project portfolio, maturity, regional distribution, outcomes, and preparedness indicators in Sub-Saharan Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at:
`https://sen.science/doi/10.71728/frontiers.kvkg-vak8/fair2.json`

In this notebook, all dataset entities such as record sets and fields are referenced by their `@id` values whenever they are available from the metadata.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn pyarrow

## 1. Data Loading
We begin by loading the Croissant metadata with `mlcroissant`. We also inspect the metadata object safely without treating it like a dictionary directly.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/frontiers.kvkg-vak8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

# Convert the metadata object to JSON for inspection while keeping dataset.metadata as a single object
metadata_json = metadata_obj.to_json()
print('Dataset name:', metadata_json.get('name', 'N/A'))
print('Dataset description:', metadata_json.get('description', 'N/A'))
print('Dataset version:', metadata_json.get('version', 'N/A'))
print('Dataset license:', metadata_json.get('license', 'N/A'))
print('Dataset identifier:', metadata_json.get('identifier', 'N/A'))

## 2. Metadata Overview
Next, we inspect the available record sets, fields, and related identifiers. Since this dataset package summary indicates that the `recordSet` list may be empty at the top level, we include checks and helper utilities to explore what is actually available from the loaded Croissant metadata.

In [ ]:
def get_recordsets_from_metadata(metadata_json):
    record_sets = metadata_json.get('recordSet', [])
    if record_sets is None:
        record_sets = []
    return record_sets

record_sets_meta = get_recordsets_from_metadata(metadata_json)
print('Number of record sets declared in metadata JSON:', len(record_sets_meta))

if record_sets_meta:
    overview_rows = []
    for rs in record_sets_meta:
        rs_id = rs.get('@id', 'N/A')
        rs_name = rs.get('name', 'N/A')
        rs_desc = rs.get('description', 'N/A')
        fields = rs.get('field', []) or []
        overview_rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'n_fields': len(fields),
            'description': rs_desc
        })
    recordset_overview_df = pd.DataFrame(overview_rows)
    display(recordset_overview_df)
else:
    print('No record sets were found in metadata_json["recordSet"].')
    print('This can happen when the dataset primarily exposes distributions/files but no explicit Croissant record sets in the top-level summary.')

In [ ]:
# Display distribution/file identifiers because they are present in this dataset package
distributions = metadata_json.get('distribution', []) or []
distribution_df = pd.DataFrame([
    {'distribution_id': d.get('@id', 'N/A')}
    for d in distributions
])
print('Number of distributions declared in metadata JSON:', len(distributions))
display(distribution_df.head(20))

### Optional detailed inspection of record set fields
If record sets are available, the following cell expands their field definitions and lists field `@id` values, names, and data types.

In [ ]:
field_rows = []

for rs in record_sets_meta:
    rs_id = rs.get('@id', 'N/A')
    rs_name = rs.get('name', 'N/A')
    for field in rs.get('field', []) or []:
        field_rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'field_id': field.get('@id', 'N/A'),
            'field_name': field.get('name', 'N/A'),
            'field_description': field.get('description', 'N/A'),
            'data_type': field.get('dataType', 'N/A'),
            'source': json.dumps(field.get('source', {})) if field.get('source') else ''
        })

fields_df = pd.DataFrame(field_rows)
if not fields_df.empty:
    display(fields_df)
else:
    print('No field definitions are available because no explicit record sets were found in the metadata summary.')

## 3. Data Access Patterns
This section follows the template style while being robust to the actual structure of this dataset.

We first try to enumerate records for each available record set by `@id`. If no record sets are present, we keep the notebook usable by exploring metadata-derived tables instead.

In [ ]:
record_set_ids = [rs.get('@id') for rs in record_sets_meta if rs.get('@id')]
print('Record set @id values:')
for rs_id in record_set_ids:
    print('-', rs_id)

sample_records_by_recordset = {}

for rs_id in record_set_ids:
    try:
        sample_records = []
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            sample_records.append(rec)
            if i >= 4:
                break
        sample_records_by_recordset[rs_id] = sample_records
        print(f'\nSample records for record set {rs_id}:')
        if sample_records:
            for rec in sample_records:
                print(rec)
        else:
            print('No records returned.')
    except Exception as e:
        print(f'Could not load records for record set {rs_id}: {e}')

if not record_set_ids:
    print('No record set IDs are available to pass to dataset.records(record_set=<@id>).')

## 4. Data Extraction into DataFrames
When record sets exist, we convert each record set into a pandas DataFrame using its `@id`.

If no record sets are available, we create DataFrames from metadata entities such as distributions and keywords so that we can still demonstrate practical exploration steps.

In [ ]:
dataframes = {}

if record_set_ids:
    for rs_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rs_id))
            dataframes[rs_id] = pd.DataFrame(records)
            print(f'Loaded {len(dataframes[rs_id])} rows for record set {rs_id}')
        except Exception as e:
            print(f'Failed to load record set {rs_id}: {e}')
else:
    # Fallback DataFrames built from metadata so the notebook remains functional
    distributions_df = pd.DataFrame([
        {'distribution_id': d.get('@id', 'N/A')}
        for d in metadata_json.get('distribution', []) or []
    ])
    keywords_df = pd.DataFrame({
        'keyword': metadata_json.get('keywords', []) or []
    })
    limitations_df = pd.DataFrame({
        'data_limitation': metadata_json.get('dataLimitations', []) or []
    })
    biases_df = pd.DataFrame({
        'data_bias': metadata_json.get('dataBiases', []) or []
    })
    use_cases_df = pd.DataFrame({
        'data_use_case': metadata_json.get('dataUseCases', []) or []
    })

    dataframes['distribution_metadata'] = distributions_df
    dataframes['keywords_metadata'] = keywords_df
    dataframes['limitations_metadata'] = limitations_df
    dataframes['biases_metadata'] = biases_df
    dataframes['use_cases_metadata'] = use_cases_df

print('Available DataFrame keys:')
print(list(dataframes.keys()))

In [ ]:
# Preview one or more extracted DataFrames
for df_name, df in dataframes.items():
    print(f'\nDataFrame: {df_name}')
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    display(df.head())

## 5. Exploratory Data Analysis (EDA)
This section demonstrates practical processing steps using the available tables.

Because this dataset summary exposes rich metadata but may not expose explicit top-level record sets, we include two EDA paths:
1. A generic EDA path for metadata-derived tables.
2. A record-set-based path that activates automatically if record sets are present.

In [ ]:
# Metadata-driven EDA: compute text lengths and basic summaries
eda_target_id = 'limitations_metadata' if 'limitations_metadata' in dataframes else None

if eda_target_id is not None:
    eda_df = dataframes[eda_target_id].copy()
    text_field_id = eda_df.columns[0]
    eda_df[f'{text_field_id}_length'] = eda_df[text_field_id].astype(str).str.len()

    threshold = eda_df[f'{text_field_id}_length'].median()
    filtered_df = eda_df[eda_df[f'{text_field_id}_length'] > threshold].copy()

    if filtered_df[f'{text_field_id}_length'].std(ddof=0) != 0:
        filtered_df[f'{text_field_id}_length_normalized'] = (
            (filtered_df[f'{text_field_id}_length'] - filtered_df[f'{text_field_id}_length'].mean()) /
            filtered_df[f'{text_field_id}_length'].std(ddof=0)
        )
    else:
        filtered_df[f'{text_field_id}_length_normalized'] = 0.0

    print(f'Filtered rows with {text_field_id}_length > {threshold}:')
    display(filtered_df)

    print('Summary statistics for text length:')
    display(eda_df[[f'{text_field_id}_length']].describe())
else:
    print('No metadata-derived limitations table available for EDA.')

In [ ]:
# Record-set-driven EDA when actual record data are available
numeric_eda_done = False

for rs_id, df in dataframes.items():
    if df.empty:
        continue
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    non_numeric_cols = [c for c in df.columns if c not in numeric_cols]
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        if filtered_df[numeric_field_id].std(ddof=0) != 0:
            filtered_df[f'{numeric_field_id}_normalized'] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
                filtered_df[numeric_field_id].std(ddof=0)
            )
        else:
            filtered_df[f'{numeric_field_id}_normalized'] = 0.0

        print(f'Using DataFrame {rs_id} for numeric EDA')
        print(f'Numeric field selected by column name: {numeric_field_id}')
        display(filtered_df.head())

        if non_numeric_cols:
            group_field_id = non_numeric_cols[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f'Grouped mean of {numeric_field_id} by {group_field_id}:')
            display(grouped_df.head())
        numeric_eda_done = True
        break

if not numeric_eda_done:
    print('No numeric columns were found in the extracted DataFrames, so numeric record-level EDA was skipped.')

## 6. Additional Dataset Summary Tables
The following cells convert selected metadata lists into compact analysis tables. This is especially useful for FAIR and governance-oriented datasets where rich documentation is part of the package itself.

In [ ]:
summary_table = pd.DataFrame([
    {'attribute': 'name', 'value': metadata_json.get('name', '')},
    {'attribute': 'identifier', 'value': metadata_json.get('identifier', '')},
    {'attribute': 'version', 'value': metadata_json.get('version', '')},
    {'attribute': 'datePublished', 'value': metadata_json.get('datePublished', '')},
    {'attribute': 'license', 'value': metadata_json.get('license', '')},
    {'attribute': 'spatialCoverage', 'value': metadata_json.get('spatialCoverage', '')},
    {'attribute': 'temporalCoverage', 'value': metadata_json.get('temporalCoverage', '')},
    {'attribute': 'conformsTo', 'value': metadata_json.get('conformsTo', '')}
])
display(summary_table)

In [ ]:
# Count lengths of list-based metadata sections
list_section_counts = pd.DataFrame([
    {'section_id': 'distribution', 'count': len(metadata_json.get('distribution', []) or [])},
    {'section_id': 'keywords', 'count': len(metadata_json.get('keywords', []) or [])},
    {'section_id': 'dataBiases', 'count': len(metadata_json.get('dataBiases', []) or [])},
    {'section_id': 'dataLimitations', 'count': len(metadata_json.get('dataLimitations', []) or [])},
    {'section_id': 'dataUseCases', 'count': len(metadata_json.get('dataUseCases', []) or [])},
    {'section_id': 'author', 'count': len(metadata_json.get('author', []) or [])},
    {'section_id': 'citation', 'count': len(metadata_json.get('citation', []) or [])},
    {'section_id': 'recordSet', 'count': len(metadata_json.get('recordSet', []) or [])}
])
display(list_section_counts)

## 7. Visualization
We now create a few simple visualizations. These plots use either extracted record sets or metadata-derived tables, depending on what is available.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: metadata section counts
plot_df = list_section_counts.sort_values('count', ascending=False)
sns.barplot(data=plot_df, x='count', y='section_id', ax=axes[0], palette='viridis')
axes[0].set_title('Counts of Metadata Sections')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Section @id/name')

# Plot 2: keyword length distribution if keywords are available
if 'keywords_metadata' in dataframes and not dataframes['keywords_metadata'].empty:
    keyword_plot_df = dataframes['keywords_metadata'].copy()
    keyword_plot_df['keyword_length'] = keyword_plot_df['keyword'].astype(str).str.len()
    sns.histplot(keyword_plot_df['keyword_length'], bins=12, kde=True, ax=axes[1], color='steelblue')
    axes[1].set_title('Distribution of Keyword Lengths')
    axes[1].set_xlabel('Keyword length')
    axes[1].set_ylabel('Frequency')
else:
    axes[1].text(0.5, 0.5, 'No keyword table available', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# Visualize text length for data limitations if available
if 'limitations_metadata' in dataframes and not dataframes['limitations_metadata'].empty:
    limitations_plot_df = dataframes['limitations_metadata'].copy()
    limitations_plot_df['length'] = limitations_plot_df['data_limitation'].astype(str).str.len()
    limitations_plot_df = limitations_plot_df.sort_values('length', ascending=True)

    plt.figure(figsize=(12, 6))
    sns.barplot(data=limitations_plot_df, x='length', y='data_limitation', palette='magma')
    plt.title('Length of Documented Data Limitations')
    plt.xlabel('Character count')
    plt.ylabel('Data limitation text')
    plt.tight_layout()
    plt.show()
else:
    print('No limitations metadata available for visualization.')

## 8. Example: Working with `@id` Values Explicitly
This cell demonstrates how to keep references to dataset entities by `@id` in variables, which is the recommended pattern for Croissant-based workflows.

In [ ]:
dataset_id = metadata_json.get('@id', 'N/A')
distribution_ids = [d.get('@id', 'N/A') for d in metadata_json.get('distribution', []) or []]
record_set_ids = [rs.get('@id', 'N/A') for rs in metadata_json.get('recordSet', []) or []]
field_ids = fields_df['field_id'].tolist() if 'fields_df' in globals() and not fields_df.empty else []

print('Dataset @id:')
print(dataset_id)
print('\nFirst 10 distribution @id values:')
print(distribution_ids[:10])
print('\nRecord set @id values:')
print(record_set_ids)
print('\nFirst 10 field @id values:')
print(field_ids[:10])

## 9. Conclusion
In this notebook, we:

- Loaded the Croissant dataset using `mlcroissant`.
- Inspected the dataset metadata through a single metadata object.
- Checked for record sets and fields using their `@id` values.
- Converted available records or metadata sections into pandas DataFrames.
- Performed simple exploratory analysis and visualization.

For this dataset, the package metadata is particularly rich and includes detailed information on distributions, limitations, biases, use cases, and governance context. If future versions expose explicit record sets in the Croissant schema, the record-set-based cells in this notebook will support loading them directly by `@id`.